# Lab 5.1 &mdash; A Support Desk That Triages Itself

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 2 &middot; Module 5 &mdash; Multi-Agent Collaboration &amp; Orchestration**

### What you'll do
- Build the supervisor/worker graph where <em>every</em> node calls the model
- Route 29 tickets &mdash; keyword traps, contested primaries, typos, and five that cannot be routed at all
- Drive the triage console, and go hunting for a ticket the router actually gets wrong
- Rewrite the three desk descriptions and watch the accuracy move without touching the graph

> **How this lab works.** You write real LangGraph code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those assert on the *objects you built*
> (a compiled `StateGraph`, a declared reducer, a routing key), so they are deterministic
> and never depend on the model. Cells marked **Run it for real** put your work in front of
> the sandbox model; that is the part worth watching. The score line is feedback, not a grade.

> **This one is a walkthrough.** There is nothing to fill in and nothing to score &mdash;
> every cell runs against the sandbox model, and the point is what the router *does* with
> awkward tickets. Labs 5.2 and 5.3 go back to blanks and self-checks.
>
> **The system on slide 2.** Tickets arrive on one queue; a supervisor reads each one and
> picks a desk; one desk answers; a `resolve` node closes it.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-5-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def _is_todo(exc: BaseException) -> bool:
    """Is this exception really an unfilled blank?

    LangGraph runs your nodes inside tasks, so the NameError from an unfilled BLANK can
    arrive wrapped. Walk the cause chain before calling anything a failure -- telling you
    your answer is wrong when you have not written one yet is the worst thing a lab does.
    """
    seen = set()
    while exc is not None and id(exc) not in seen:
        if isinstance(exc, NameError):
            return True
        seen.add(id(exc))
        exc = exc.__cause__ or exc.__context__
    return False

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except Exception as exc:
        if _is_todo(exc):
            print(f"[TODO] {name}")
            _results.append(None)
            return
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except Exception as exc:
        if not _is_todo(exc):
            raise
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Off is the default here because the "Run it for real" cells make many small calls.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# A customer support desk for a SaaS product. One queue in, three specialists behind it.
# This case file is Module 5 lab 5.1 only -- 5.2 and 5.3 are different systems.

SPECIALISTS = ("billing", "tech", "account")

# What the supervisor gets to read. These three lines ARE the router's program -- there is
# no other instruction anywhere. You will edit them at the end of the lab and re-measure.
DESK = {
    "billing": "Charges, refunds, invoices, plan and price changes.",
    "tech":    "Errors, outages, failing API calls, anything broken.",
    "account": "Seats, owners, permissions, sign-in and access.",
}

# How each specialist answers once it has the ticket.
PERSONA = {
    "billing": "You are the billing desk of a SaaS company. Answer in one sentence, plainly, "
               "and name the next concrete step. Never promise a refund amount.",
    "tech":    "You are the technical support desk of a SaaS company. Answer in one sentence, "
               "ask for the one diagnostic detail you most need, and never guess a root cause.",
    "account": "You are the account desk of a SaaS company. Answer in one sentence and say "
               "who has to authorise the change. Never change access on the asker's word alone.",
}

# 29 tickets to play with. `expected` is the desk a human would pick; five of them are
# deliberately unroutable and carry None, so they are excluded from any accuracy.
SCENARIOS = [
    # (ticket, expected desk or None, why this one is in the set)
    ("I was charged twice for March.",                                    "billing", "plain"),
    ("Can I get an invoice with our VAT number on it?",                   "billing", "plain"),
    ("We want to downgrade to the starter plan.",                         "billing", "plain"),
    ("Your API returns 500 on every /sync call since 09:00.",             "tech",    "plain"),
    ("The export button throws an error and nothing downloads.",          "tech",    "plain"),
    ("Webhooks stopped firing after your deploy.",                        "tech",    "plain"),
    ("Please add two more seats for the new joiners.",                    "account", "plain"),
    ("Move the workspace owner to priya@example.com.",                    "account", "plain"),

    # the loudest word points at one desk and the problem belongs to another
    ("The invoice page throws a 500.",                                    "tech",    "keyword trap"),
    ("I cannot open billing settings -- it says permission denied.",      "account", "keyword trap"),
    ("Our refund never arrived and the support chat is down too.",        "billing", "keyword trap"),

    # the intent is only implied; no word in the ticket names the desk
    ("Nobody on my team can get in this morning.",                        "account", "implied"),
    ("We were told this would be free until June.",                       "billing", "implied"),
    ("Everything was fine yesterday and now nothing loads.",              "tech",    "implied"),
    ("Someone who left in May can still see our data.",                   "account", "implied"),

    # written the way people actually write to a support desk
    ("cant login sicne mornign, urgnt",                                   "account", "typos"),
    ("Hi! Hope you are well. Quick one -- can we switch to annual billing?",
                                                                          "billing", "buried in politeness"),
    ("This is the third time. Cancel everything and refund us.",          "billing", "angry"),

    # the desk turns on which fact is the PROBLEM, not on which words are present
    ("Can you confirm the seat count on our last invoice?",               "billing", "contested primary"),
    ("We are being billed for a user we deleted in April.",               "billing", "contested primary"),
    ("Why was my card declined when I tried to add a seat?",              "billing", "contested primary"),
    ("The 500 error only happens for users on the free plan.",            "tech",    "contested primary"),
    ("Our SSO broke right after you changed the pricing page.",           "tech",    "contested primary"),
    ("The new admin cannot approve invoices -- she has no such button.",  "account", "contested primary"),

    # no single defensible answer -- these are the interesting ones
    ("It is not working.",                                                None, "too vague to route"),
    ("I was charged for seats we never got and now I cannot log in either.",
                                                                          None, "two desks at once"),
    ("Do you sponsor conferences?",                                       None, "not a support ticket"),
    ("Renewal is next week and our admin's SSO login is broken.",         None, "two desks at once"),
    ("Please remove the card on file and delete the workspace.",          None, "two desks at once"),
]

SCORED = [(t, e) for t, e, _ in SCENARIOS if e is not None]
print(f"{len(SCENARIOS)} scenarios ({len(SCORED)} with a defensible answer), "
      f"{len(DESK)} desks")

## Concept

A supervisor decides which specialist handles a request. In LangGraph that is one thing: a
**conditional edge** out of a supervisor node.

`add_conditional_edges(source, fn, path_map)` needs two different things, and people mix them up:

| | |
|---|---|
| `fn` | a function of **state** that returns a **key** |
| `path_map` | `{key: node name}` &mdash; which node each key means |

Which makes the supervisor a classifier with a known correct answer. So it has an accuracy, and
almost nobody measures it &mdash; even though a misroute wastes every token spent downstream of it.

The thing to watch for in this lab: the supervisor's entire program is the three sentences in
`DESK`. There is no other instruction anywhere in the system.

## Section 1 &mdash; The graph, with the model in every node

The supervisor is a model call. So is each desk. Nothing here is stubbed, which is why every
cell in this notebook is a *Run it for real* cell.

In [ ]:
# ---------------------------------------------- Run it for real: the graph, model and all
# Every node here calls the model. The supervisor decides which desk; the desk answers.
# That is the whole system on slide 2, and there is nothing stubbed in it.

from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END


class DeskState(TypedDict):
    ticket: str                       # what the customer wrote
    route: str | None                 # the supervisor's decision, readable afterwards
    why: str | None                   # and its reason, in its own words
    answer: str | None                # the specialist's reply
    trail: Annotated[list, add]       # append: every node leaves a mark


def desk_listing(desks: dict) -> str:
    return "\n".join(f"- {name}: {text}" for name, text in desks.items())


def ask_the_router(ticket: str, desks: dict | None = None) -> tuple[str, str]:
    """One model call. Returns (desk, why).

    Two plain lines rather than with_structured_output, deliberately: on this gateway the
    schema route is several times slower, and an app you are clicking through has to feel
    like an app. The last cell times both on your own run -- do not take the ratio on trust,
    it moves with load.
    """
    desks = desks or DESK
    system = ("You route one customer support ticket to exactly one desk.\n"
              f"{desk_listing(desks)}\n"
              "Answer with exactly two lines and nothing else:\n"
              "desk: <one of " + ", ".join(desks) + ">\n"
              "why: <at most 10 words>")
    reply = ask(ticket, system=system) or ""
    desk, why = None, reply.strip().replace("\n", " ")[:90]
    for line in reply.splitlines():
        low = line.strip().lower()
        if low.startswith("desk:"):
            word = low[5:].strip().strip("`*.\"' ")
            desk = word if word in desks else None
        elif low.startswith("why:"):
            why = line.strip()[4:].strip()
    if desk is None:                                  # the model ignored the format
        desk = next((d for d in desks if d in reply.lower()), list(desks)[0])
        why = "(unparsed reply -- fell back to the first desk named)"
    return desk, why


def supervisor(state: DeskState) -> dict:
    """A node like any other. It decides, and it writes the decision down."""
    desk, why = ask_the_router(state["ticket"])
    return {"route": desk, "why": why, "trail": [f"supervisor -> {desk}"]}


def make_specialist(name: str):
    """Three desks that differ only in the system prompt they answer with."""
    def specialist(state: DeskState) -> dict:
        reply = ask(f"Ticket: {state['ticket']}", system=PERSONA[name])
        return {"answer": reply, "trail": [f"{name} answered"]}
    return specialist


def resolve(state: DeskState) -> dict:
    return {"trail": ["resolved"]}


def pick_specialist(state: DeskState) -> str:
    """The adapter a conditional edge needs: takes STATE, returns a KEY of the path map."""
    return state["route"]


def build_desk():
    g = StateGraph(DeskState)
    g.add_node("supervisor", supervisor)
    for name in SPECIALISTS:
        g.add_node(name, make_specialist(name))
    g.add_node("resolve", resolve)

    g.add_edge(START, "supervisor")
    g.add_conditional_edges("supervisor", pick_specialist, {n: n for n in SPECIALISTS})
    for name in SPECIALISTS:
        g.add_edge(name, "resolve")
    g.add_edge("resolve", END)
    return g.compile()


def fresh(ticket: str) -> dict:
    return {"ticket": ticket, "route": None, "why": None, "answer": None, "trail": []}


print("graph: START -> supervisor -> [ billing | tech | account ] -> resolve -> END")

In [ ]:
# ---------------------------------------------- Run it for real: one ticket, end to end
def route_ticket(ticket: str) -> dict:
    """Run one ticket through the whole graph and print what each node did."""
    if not llm_ready():
        return {}
    result = build_desk().invoke(fresh(ticket))
    print(f'  ticket      "{ticket}"')
    print(f"  supervisor  -> {result['route']}   ({result['why']})")
    print(f"  {result['route']} desk replies:")
    print(textwrap.fill(result["answer"] or "", 92,
                        initial_indent="    ", subsequent_indent="    "))
    print(f"  trail       {result['trail']}")
    return result

route_ticket("The invoice page throws a 500.")

## Section 2 &mdash; Twenty-one tickets, and three of them have no right answer

Eight plain ones; three where the loudest word points at the wrong desk; four whose intent is
only implied; three written the way people actually write; six where two desks are both
mentioned and the answer turns on which fact is the *problem* &mdash; and five that genuinely
cannot be routed: too vague, two desks at once, and one that is not a support ticket at all.

Those last five are excluded from the accuracy. Watch what the router does with them anyway:
it will answer confidently, because nothing in the prompt gives it permission not to.

In [ ]:
# ---------------------------------------------- Run it for real: all 21 scenarios
# Only the SUPERVISOR runs here -- routing is what is being measured, and waking a desk
# for every scenario would triple the wall clock without changing a single decision.

def sweep(desks: dict | None = None, quiet: bool = False) -> dict:
    """Route every scenario. Returns {ticket: (desk, why)}."""
    out = {}
    for ticket, expected, tag in SCENARIOS:
        desk, why = ask_the_router(ticket, desks)
        out[ticket] = (desk, why)
        if not quiet:
            mark = " " if expected is None else ("ok" if desk == expected else "XX")
            want = expected or "--"
            print(f"  {mark}  {desk:8} (wanted {want:8}) [{tag:20}] {ticket[:44]}")
    return out


def accuracy(routed: dict) -> float:
    """Over the scenarios that HAVE a defensible answer. The other three are not failures."""
    return sum(1 for t, e in SCORED if routed.get(t, (None,))[0] == e) / len(SCORED)


def confusion(routed: dict) -> dict:
    out = {}
    for ticket, expected in SCORED:
        got = routed.get(ticket, (None,))[0]
        if got != expected:
            out[(expected, got)] = out.get((expected, got), 0) + 1
    return out


if llm_ready():
    ROUTED = sweep()
    print(f"\n  accuracy on the {len(SCORED)} scorable tickets: {accuracy(ROUTED):.0%}")
    for (want, got), n in sorted(confusion(ROUTED).items(), key=lambda kv: -kv[1]):
        print(f"    {n}x  should have been {want:8} -> went to {got}")
    print("\n  and the ones with no defensible answer:")
    for ticket, expected, tag in SCENARIOS:
        if expected is None:
            print(f"    {ROUTED[ticket][0]:8} <- {ticket[:52]}   ({tag})")
    print("\n  It answered every one of them, confidently. Nothing in the prompt gave it")
    print("  permission to say 'I do not know' -- which is a design choice you made by omission.")
    if accuracy(ROUTED) > 0.95:
        print("\n  Note the score. A model supervisor with three good descriptions is very")
        print("  hard to beat on a queue like this, and an eval set your system already passes")
        print("  is not an eval set any more -- it is a regression test. The console below is")
        print("  how you go and find the tickets it does get wrong. Those are the keepers.")

## Section 3 &mdash; The triage console

Pick a scenario or type your own. Route it, read what the desk says back, and label it &mdash;
that is how `MY_EVAL` fills up, and a ticket you disagreed with is worth ten you did not.

**The exercise: find a ticket this router gets wrong.** It is harder than it looks, and that is
the point &mdash; the twenty-nine above did not manage it. Things that tend to work: a ticket
whose problem belongs to one desk and whose *urgency* belongs to another; a ticket quoting an
error message from some other product; a ticket in a language the descriptions are not written
in; a ticket where the customer has already diagnosed it themselves, wrongly.

When you find one, log it. That ticket is now worth more than the whole starter set, because it
is the only one that can tell you whether tomorrow's change made things better or worse.

In [ ]:
# ---------------------------------------------- Run it for real: the triage console
# A small app. Pick a scenario or type your own, watch the graph route it and the desk
# answer, and when it goes to the wrong desk say so -- that is how the eval set gets built.

MY_EVAL = []          # (ticket, the desk you say it should be, the desk it chose)


def triage_console():
    try:
        import ipywidgets as W
        from IPython.display import display, clear_output
    except ImportError:
        print("ipywidgets is not available here. Use route_ticket('your ticket') instead.")
        return

    picker = W.Dropdown(
        options=[("-- type your own below --", "")] +
                [(f"[{tag}]  {t[:56]}", t) for t, _, tag in SCENARIOS],
        layout=W.Layout(width="780px"))
    text = W.Textarea(value=SCENARIOS[0][0], placeholder="type a support ticket",
                      layout=W.Layout(width="780px", height="62px"))
    go = W.Button(description="Route it", button_style="primary")
    seen = W.Button(description="My eval set", layout=W.Layout(width="150px"))
    should = W.Dropdown(options=[("should have been...", None)] + [(d, d) for d in SPECIALISTS],
                        layout=W.Layout(width="220px"))
    log = W.Button(description="Log that", layout=W.Layout(width="130px"))
    out = W.Output()
    last = {"ticket": None, "chose": None}

    picker.observe(lambda c: c["new"] and setattr(text, "value", c["new"]), names="value")

    def on_go(_):
        with out:
            clear_output()
            if not llm_ready():
                return
            try:
                r = build_desk().invoke(fresh(text.value))
            except Exception as exc:
                print(f"  the graph raised {type(exc).__name__}: {exc}")
                return
            last.update(ticket=text.value, chose=r["route"])
            print(f"  supervisor -> {r['route']}    ({r['why']})")
            print(f"  the {r['route']} desk replies:\n")
            print(textwrap.fill(r["answer"] or "", 90,
                                initial_indent="    ", subsequent_indent="    "))

    def on_log(_):
        with out:
            if not last["ticket"]:
                print("\n  route a ticket first.")
            elif should.value is None:
                print("\n  pick the desk it should have gone to, then press Log that.")
            else:
                MY_EVAL.append((last["ticket"], should.value, last["chose"]))
                verdict = "agreed" if should.value == last["chose"] else "MISROUTE"
                print(f"\n  logged ({verdict}): {last['chose']} -> should be {should.value}"
                      f"    [{len(MY_EVAL)} in your eval set]")

    def on_seen(_):
        with out:
            clear_output()
            if not MY_EVAL:
                print("  nothing logged yet -- route a few and correct the ones it gets wrong.")
                return
            for ticket, want, got in MY_EVAL:
                mark = "ok" if want == got else "XX"
                print(f"  {mark}  wanted {want:8} got {got:8}  {ticket[:56]}")
            miss = sum(1 for _, w, g in MY_EVAL if w != g)
            print(f"\n  {miss} misroute(s) in {len(MY_EVAL)} labelled tickets"
                  f"  ({1 - miss / len(MY_EVAL):.0%} accurate on YOUR eval set)")

    go.on_click(on_go)
    log.on_click(on_log)
    seen.on_click(on_seen)
    display(W.VBox([picker, text, W.HBox([go, seen]), W.HBox([should, log]), out]))


triage_console()

## Section 4 &mdash; The descriptions are the router

Same model, same graph, same tickets. Three different sets of desk descriptions.

This is the Module 1 tool-description A/B one layer up: there, wording changed which *tool* an
agent picked; here it changes which *agent* the work goes to, and the blast radius is a whole
downstream conversation rather than one call.

In [ ]:
# ---------------------------------------------- Run it for real: the descriptions ARE the router
# Same model, same graph, same 21 scenarios. The only thing that changes is the three lines
# the supervisor reads. This is the A/B from Module 1's tool descriptions, one layer up.

VAGUE = {
    "billing": "Money things.",
    "tech":    "Technical things.",
    "account": "Account things.",
}

SHARPER = {
    "billing": ("Anything about money that has already moved or is about to: a charge, a "
                "refund, an invoice, a price, a plan change, a contract term someone was "
                "promised. Route here when the customer's loss is financial."),
    "tech":    ("Anything the product is doing wrong: an error, a 500, a page that will not "
                "load, an integration that stopped, a deploy that broke something. Route "
                "here when something that used to work does not. A billing PAGE that errors "
                "is a tech ticket."),
    "account": ("Anything about who may do what: seats, owners, roles, permissions, sign-in, "
                "offboarding, access that should or should not exist. Route here when the "
                "answer is about a person rather than about money or a bug."),
}

if llm_ready():
    print("three one-line descriptions (the original):")
    base = ROUTED if "ROUTED" in dir() else sweep(quiet=True)
    print(f"  accuracy {accuracy(base):.0%}\n")
    for label, desks in (("vague", VAGUE), ("sharper", SHARPER)):
        r = sweep(desks, quiet=True)
        print(f"{label} descriptions:")
        print(f"  accuracy {accuracy(r):.0%}")
        for (want, got), n in sorted(confusion(r).items(), key=lambda kv: -kv[1]):
            print(f"    {n}x  {want} -> {got}")
        print()
    print("  Nothing about the model or the graph changed between those three runs.")

## Section 5 &mdash; And why the router does not use a schema

Worth knowing before you reach for `with_structured_output` in something a person is waiting on.

In [ ]:
# ---------------------------------------------- Run it for real: why not with_structured_output?
# The framework way to get a field out of a model is a schema. Here is what it costs on this
# gateway, on the same three tickets, so the choice in ask_the_router is one you can check.

def route_with_schema(tickets: list[str]) -> tuple[list, float]:
    """The framework way: a Pydantic schema and with_structured_output."""
    from pydantic import BaseModel, Field
    class Routing(BaseModel):
        desk: str = Field(description="one of: billing, tech, account")
        why:  str = Field(description="at most 10 words")
    router = get_llm().with_structured_output(Routing)
    system = "Route one customer support ticket to exactly one desk.\n" + desk_listing(DESK)
    t0 = time.time()
    picks = []
    for t in tickets:
        r = router.invoke([("system", system), ("human", t)])
        picks.append(None if r is None else r.desk)
    return picks, time.time() - t0


def route_with_two_lines(tickets: list[str]) -> tuple[list, float]:
    t0 = time.time()
    picks = [ask_the_router(t)[0] for t in tickets]
    return picks, time.time() - t0


if llm_ready():
    hard = [t for t, _, tag in SCENARIOS
            if tag in ("keyword trap", "contested primary", "too vague to route",
                       "two desks at once")]
    sample = hard[:8]
    schema_picks, schema_s = route_with_schema(sample)
    plain_picks, plain_s = route_with_two_lines(sample)

    agreed = sum(1 for a, b in zip(schema_picks, plain_picks) if a == b)
    print(f"{'ticket':46}{'schema':>10}{'two lines':>12}")
    print("-" * 68)
    for t, a, b in zip(sample, schema_picks, plain_picks):
        print(f"{t[:44]:46}{str(a):>10}{str(b):>12}" + ("" if a == b else "   <- differ"))
    print(f"\n  with_structured_output  {schema_s:5.1f}s")
    print(f"  two plain lines         {plain_s:5.1f}s   ({schema_s / plain_s:.1f}x faster)")
    print(f"  they agreed on {agreed}/{len(sample)} of these tickets")
    print("\n  A schema is guided decoding, and guided decoding is not free. Use it when the")
    print("  SHAPE matters more than the latency -- Lab 5.2's findings, not an app you click.")
    print("  Note the disagreements, if you got any: the schema call and the two-line call")
    print("  are different prompts, so this is not a pure latency comparison and you should")
    print("  not read it as one.")

## Your turn

1. Add a fourth desk &mdash; `none` &mdash; and give it a description that says what does *not*
   belong on this queue. Re-run the sweep. The three unroutable tickets should move; check
   whether anything else moved with them, because a new option changes every decision, not just
   the ones you meant it to.
2. Make the supervisor say how sure it is, and route anything below your threshold to a human.
   You now have to pick the threshold, and `MY_EVAL` is the only evidence you have for it.
3. The desks answer without ever reading the ticket's history, the account, or the ledger. Give
   one of them a tool and watch the reply change &mdash; that is Module 4's work arriving inside
   Module 5's graph.
4. Time the console end to end. Two model calls per ticket is not free: decide out loud whether
   a first-line desk should route with a small model and answer with a large one.